[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LinkedInLearning/Fine-Tuning-for-LLMs-from-Beginner-to-Advanced/blob/main/CH05/05_01_end/notebooks/05_01_configure_lora.ipynb)

# 05_01 Configure LoRA hyperparameters

Set up the LoRA adapter and the training settings, and see what each one controls. You'll train with these settings in the next video.

Before you start, choose **Runtime > Change runtime type > T4 GPU**.

In [ ]:
%%capture
# Install Unsloth. This takes a minute or two.
!pip install unsloth

In [ ]:
# Pull the course files into this Colab session and load the Ridgeline helpers.
!git clone --quiet https://github.com/LinkedInLearning/Fine-Tuning-for-LLMs-from-Beginner-to-Advanced.git course 2>/dev/null || echo "Course files already downloaded."

import sys
sys.path.insert(0, "course/CH05")

import ridgeline

## Load the model in 4-bit

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3-4B-Instruct-2507",
    max_seq_length=2048,
    load_in_4bit=True,
)

## Add the LoRA adapter

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                   # rank: how much room the adapter has to learn
    lora_alpha=16,          # scaling: how strongly the adapter's update counts
    lora_dropout=0,         # Unsloth is optimized for 0
    target_modules=[        # the layers that get adapters
        "q_proj", "k_proj", "v_proj", "o_proj",   # attention
        "gate_proj", "up_proj", "down_proj",      # feed-forward
    ],
    bias="none",
    use_gradient_checkpointing="unsloth",       # saves memory
    random_state=3407,
)
model.print_trainable_parameters()

## See how rank changes the adapter size

In [ ]:
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

# LoRA weights grow in step with the rank, so you can estimate other ranks from this one.
for rank in [8, 16, 32, 64]:
    print(f"rank {rank:>2}: about {trainable * rank / 16 / 1e6:.0f} million trainable weights")

## Set the training settings

In [ ]:
from trl import SFTConfig
from unsloth import is_bfloat16_supported

config = SFTConfig(
    output_dir="outputs",
    dataset_text_field="text",
    max_length=2048,
    per_device_train_batch_size=2,     # examples per step on the GPU
    gradient_accumulation_steps=4,     # 2 x 4 = 8 examples per update
    num_train_epochs=2,               # passes through the training data
    learning_rate=2e-4,              # how big each update step is
    warmup_steps=5,                    # ease into the learning rate
    lr_scheduler_type="linear",
    optim="adamw_8bit",                # memory-efficient optimizer
    weight_decay=0.01,
    logging_steps=5,
    eval_strategy="steps",
    eval_steps=20,
    per_device_eval_batch_size=4,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    seed=3407,
    report_to="none",
)

In [ ]:
effective = config.per_device_train_batch_size * config.gradient_accumulation_steps
print(f"Examples per update: {effective}")
print(f"Learning rate: {config.learning_rate}")
print(f"Epochs: {config.num_train_epochs}")

## Starting points that work for most projects

| Setting | Start here | Raise it when | Lower it when |
| --- | --- | --- | --- |
| Rank `r` | 16 | The model isn't picking up the pattern | You're short on memory |
| `lora_alpha` | Same as `r` | You want the adapter to count for more | The model changes too much |
| Learning rate | 2e-4 | Loss barely moves | Loss jumps around or spikes |
| Epochs | 1 to 3 | Evaluation loss is still falling at the end | Evaluation loss starts rising |